# DeliveryIQ — Stage 1: Dataset Audit & Exploratory Data Analysis

**Scope:** read-only audit of `data/raw/train.csv`. No cleaning is persisted, no rows are removed,
no models are trained. All parsing below happens **in memory** for inspection only.

**Outputs**
- `figures/*.png` — audit charts
- `reports/audit_metrics.json` — machine-readable numbers quoted in `reports/dataset_audit_report.md`

The raw file's SHA-256 is checked at the start and the end to prove it was not modified.

In [1]:
import hashlib, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "train.csv"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
REP = ROOT / "reports"; REP.mkdir(exist_ok=True)
SEED = 42

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW_SHA_START = sha256(RAW)
print("Raw file:", RAW, "| bytes:", RAW.stat().st_size, "| sha256:", RAW_SHA_START)

# ---- chart style (reference palette from the dataviz method) ----
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f3f7fd", "#9cc3f0", BLUE, "#123f78"])
DIV = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlecolor": INK, "font.size": 10, "figure.dpi": 110,
})
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved figures/" + name)

METRICS = {}

Raw file: D:\Project1\data\raw\train.csv | bytes: 7772212 | sha256: d293b6fc7157aef30fb34301216a3432ec895d407ce2ea9edddf942b416cdde1


## 1. Load raw data (every column as string, no implicit NA conversion)

In [2]:
raw = pd.read_csv(RAW, dtype=str, keep_default_na=False)
print("rows, cols:", raw.shape)
print("columns:", list(raw.columns))
METRICS["shape"] = {"rows": int(raw.shape[0]), "cols": int(raw.shape[1])}
METRICS["columns"] = list(raw.columns)
raw.head()

rows, cols: (45593, 20)
columns: ['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)']


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.6784,12.924264,77.6884,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


## 2. Formatting & hidden missing values
Pandas reports zero nulls on the raw file because missing values are stored as **text tokens**
(`"NaN "` with a trailing space, and `"conditions NaN"` in the weather column). Many columns also
carry trailing whitespace.

In [3]:
fmt = []
for c in raw.columns:
    s = raw[c]; st = s.str.strip()
    token_nan = (st == "NaN") | (st == "conditions NaN")
    fmt.append({"column": c, "pandas_isna": int(s.isna().sum()),
                "trailing_or_leading_ws": int((s != st).sum()),
                "text_missing_tokens": int(token_nan.sum()),
                "missing_pct": round(100 * token_nan.mean(), 2),
                "n_unique_raw": int(s.nunique())})
fmt = pd.DataFrame(fmt).set_index("column")
METRICS["format"] = fmt.reset_index().to_dict("records")
fmt

,pandas_isna,trailing_or_leading_ws,text_missing_tokens,missing_pct,n_unique_raw
column,,,,,
ID,0,45593,0,0.00,45593
Delivery_person_ID,0,45593,0,0.00,1320
Delivery_person_Age,0,1854,1854,4.07,23
Delivery_person_Ratings,0,1908,1908,4.18,29
Restaurant_latitude,0,0,0,0.00,658
Restaurant_longitude,0,0,0,0.00,519
Delivery_location_latitude,0,0,0,0.00,4373
Delivery_location_longitude,0,0,0,0.00,4373
Order_Date,0,0,0,0.00,44


## 3. In-memory typed view (inspection only — not saved)

In [4]:
d = raw.apply(lambda s: s.str.strip())
d["Weatherconditions"] = d["Weatherconditions"].str.replace("conditions ", "", regex=False)
d = d.replace({"NaN": np.nan})
d["target_min"] = d["Time_taken(min)"].str.replace("(min)", "", regex=False).str.strip()
assert d["target_min"].str.fullmatch(r"\d+").all(), "unexpected target format"
d["target_min"] = d["target_min"].astype(int)

NUM = ["Delivery_person_Age", "Delivery_person_Ratings", "Restaurant_latitude", "Restaurant_longitude",
       "Delivery_location_latitude", "Delivery_location_longitude", "Vehicle_condition", "multiple_deliveries"]
for c in NUM:
    d[c] = pd.to_numeric(d[c], errors="raise")
CAT = ["Weatherconditions", "Road_traffic_density", "Type_of_order", "Type_of_vehicle", "Festival", "City"]

d["order_date"] = pd.to_datetime(d["Order_Date"], format="%d-%m-%Y", errors="coerce")
d["t_ordered"] = pd.to_datetime(d["Time_Orderd"], format="%H:%M:%S", errors="coerce")
d["t_picked"] = pd.to_datetime(d["Time_Order_picked"], format="%H:%M:%S", errors="coerce")
print("Order_Date parse failures:", int(d["order_date"].isna().sum()))
print("Time_Orderd unparsable (all are 'NaN' tokens):", int(d["t_ordered"].isna().sum()))
print("Time_Order_picked parse failures:", int(d["t_picked"].isna().sum()))
d.dtypes

Order_Date parse failures: 0
Time_Orderd unparsable (all are 'NaN' tokens): 1731
Time_Order_picked parse failures: 0


ID                                        str
Delivery_person_ID                        str
Delivery_person_Age                   float64
Delivery_person_Ratings               float64
Restaurant_latitude                   float64
Restaurant_longitude                  float64
Delivery_location_latitude            float64
Delivery_location_longitude           float64
Order_Date                                str
Time_Orderd                               str
Time_Order_picked                         str
Weatherconditions                         str
Road_traffic_density                      str
Vehicle_condition                       int64
Type_of_order                             str
Type_of_vehicle                           str
multiple_deliveries                   float64
Festival                                  str
City                                      str
Time_taken(min)                           str
target_min                              int64
order_date                     dat

## 4. Missing values

In [5]:
miss = d[raw.columns.drop("Time_taken(min)")].isna().sum().rename("missing").to_frame()
miss["pct"] = (100 * miss["missing"] / len(d)).round(2)
miss = miss[miss.missing > 0].sort_values("missing", ascending=False)
rows_any = int(d[miss.index].isna().any(axis=1).sum())
print("rows with >=1 missing field:", rows_any, f"({100*rows_any/len(d):.2f}%)")
METRICS["missing"] = miss.reset_index(names="column").to_dict("records")
METRICS["rows_any_missing"] = rows_any

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(miss.index[::-1], miss["pct"][::-1], color=BLUE, height=0.6)
for y, (n, p) in enumerate(zip(miss["missing"][::-1], miss["pct"][::-1])):
    ax.text(p + 0.05, y, f"{n:,} ({p:.2f}%)", va="center", color=INK2, fontsize=9)
ax.set_xlabel("% of rows missing"); ax.set_xlim(0, miss["pct"].max() * 1.35)
ax.set_title("Missing values per column (text 'NaN' tokens)"); ax.grid(axis="y", visible=False)
save(fig, "01_missing_values.png")
miss

rows with >=1 missing field: 4225 (9.27%)


saved figures/01_missing_values.png


,missing,pct
Delivery_person_Ratings,1908,4.18
Delivery_person_Age,1854,4.07
Time_Orderd,1731,3.80
City,1200,2.63
multiple_deliveries,993,2.18
Weatherconditions,616,1.35
Road_traffic_density,601,1.32
Festival,228,0.50


### 4b. Is missingness random? Co-occurrence and target by missingness

In [6]:
mcols = list(miss.index)
co = d[mcols].isna().astype(int).corr().round(2)
both_age_rating = int((d.Delivery_person_Age.isna() & d.Delivery_person_Ratings.isna()).sum())
fest_missing_target = d.loc[d.Festival.isna(), "target_min"].describe().round(2)
print("Age & Rating missing together:", both_age_rating)
print("Target when Festival is missing:\n", fest_missing_target)
tbl = pd.DataFrame({c: [d.loc[d[c].isna(), "target_min"].mean(), d.loc[d[c].notna(), "target_min"].mean()]
                    for c in mcols}, index=["target_mean_if_missing", "target_mean_if_present"]).T.round(2)
METRICS["missing_cooccurrence"] = co.to_dict()
METRICS["age_rating_missing_together"] = both_age_rating
METRICS["target_by_missingness"] = tbl.to_dict("index")

fig, ax = plt.subplots(figsize=(7, 5.6))
sns.heatmap(co, annot=True, fmt=".2f", cmap=SEQ, vmin=0, vmax=1, square=True, cbar_kws={"shrink": .7},
            linewidths=2, linecolor=SURF, ax=ax)
ax.set_title("Missingness co-occurrence (correlation of null indicators)")
save(fig, "02_missing_cooccurrence.png")
tbl

Age & Rating missing together: 1854
Target when Festival is missing:
 count    228.00
mean      11.17
std        1.23
min       10.00
25%       10.00
50%       11.00
75%       12.00
max       14.00
Name: target_min, dtype: float64


saved figures/02_missing_cooccurrence.png


,target_mean_if_missing,target_mean_if_present
Delivery_person_Ratings,26.42,26.29
Delivery_person_Age,26.46,26.29
Time_Orderd,26.31,26.29
City,22.06,26.41
multiple_deliveries,22.77,26.37
Weatherconditions,26.55,26.29
Road_traffic_density,26.54,26.29
Festival,11.17,26.37


## 5. Duplicates

In [7]:
dups = {"full_row_duplicates": int(raw.duplicated().sum()),
        "duplicates_ignoring_ID": int(raw.drop(columns="ID").duplicated().sum()),
        "duplicate_IDs": int(raw["ID"].duplicated().sum())}
METRICS["duplicates"] = dups
dups

{'full_row_duplicates': 0, 'duplicates_ignoring_ID': 0, 'duplicate_IDs': 0}

## 6. Cardinality & data types

In [8]:
card = pd.DataFrame({"raw_dtype": raw.dtypes.astype(str),
                     "n_unique (cleaned)": [d[c].nunique() for c in raw.columns],
                     "example": [d[c].dropna().iloc[0] for c in raw.columns]})
METRICS["cardinality"] = {c: int(d[c].nunique()) for c in raw.columns}
card

,raw_dtype,n_unique (cleaned),example
ID,str,45593,0x4607
Delivery_person_ID,str,1320,INDORES13DEL02
Delivery_person_Age,str,22,37.0
Delivery_person_Ratings,str,28,4.9
Restaurant_latitude,str,657,22.745049
Restaurant_longitude,str,518,75.892471
Delivery_location_latitude,str,4373,22.765049
Delivery_location_longitude,str,4373,75.912471
Order_Date,str,44,19-03-2022
Time_Orderd,str,176,11:30:00


## 7. Numerical descriptive statistics

In [9]:
desc = d[NUM + ["target_min"]].describe().T.round(3)
METRICS["numeric_describe"] = desc.to_dict("index")
print("Age value counts (ends):", d.Delivery_person_Age.value_counts().sort_index().iloc[[0, 1, -2, -1]].to_dict())
print("Rating value counts (ends):", d.Delivery_person_Ratings.value_counts().sort_index().iloc[[0, 1, -2, -1]].to_dict())
desc

Age value counts (ends): {15.0: 38, 20.0: 2136, 39.0: 2144, 50.0: 53}
Rating value counts (ends): {1.0: 38, 2.5: 20, 5.0: 3996, 6.0: 53}


,count,mean,std,min,25%,50%,75%,max
Delivery_person_Age,43739.0,29.567,5.815,15.000,25.000,30.000,35.000,50.000
Delivery_person_Ratings,43685.0,4.634,0.335,1.000,4.500,4.700,4.900,6.000
Restaurant_latitude,45593.0,17.018,8.185,-30.906,12.933,18.547,22.728,30.914
Restaurant_longitude,45593.0,70.231,22.884,-88.366,73.170,75.898,78.044,88.433
Delivery_location_latitude,45593.0,17.465,7.335,0.010,12.988,18.634,22.785,31.054
Delivery_location_longitude,45593.0,70.846,21.119,0.010,73.280,76.003,78.107,88.563
Vehicle_condition,45593.0,1.023,0.839,0.000,0.000,1.000,2.000,3.000
multiple_deliveries,44600.0,0.745,0.572,0.000,0.000,1.000,1.000,3.000
target_min,45593.0,26.295,9.384,10.000,19.000,26.000,32.000,54.000


In [10]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, c in zip(axes.flat, ["Delivery_person_Age", "Delivery_person_Ratings", "Vehicle_condition",
                             "multiple_deliveries", "Restaurant_latitude", "Restaurant_longitude"]):
    v = d[c].dropna()
    if v.nunique() <= 30:
        vc = v.value_counts().sort_index()
        ax.bar([f"{x:g}" for x in vc.index], vc.values, color=BLUE, width=0.7)
        if vc.size > 10:
            ax.set_xticks(range(0, vc.size, max(1, vc.size // 8)))
    else:
        ax.hist(v, bins=50, color=BLUE, edgecolor=SURF, linewidth=0.5)
    ax.set_title(c); ax.set_ylabel("rows")
fig.suptitle("Numerical feature distributions (raw values, missing excluded)", fontweight="bold", color=INK)
save(fig, "04_numeric_distributions.png")

saved figures/04_numeric_distributions.png


## 8. Categorical distributions

In [11]:
cat_counts = {c: d[c].fillna("<missing>").value_counts().to_dict() for c in CAT}
METRICS["categorical_counts"] = cat_counts
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, c in zip(axes.flat, CAT):
    vc = d[c].fillna("<missing>").value_counts()
    colors = [GRAY if k == "<missing>" else BLUE for k in vc.index]
    ax.barh(vc.index[::-1], vc.values[::-1], color=colors[::-1], height=0.6)
    ax.set_title(c); ax.set_xlabel("rows"); ax.grid(axis="y", visible=False)
fig.suptitle("Categorical distributions (gray = missing token)", fontweight="bold", color=INK)
save(fig, "05_categorical_distributions.png")
pd.Series({c: len(v) for c, v in cat_counts.items()}, name="levels incl. missing")

saved figures/05_categorical_distributions.png


Weatherconditions       7
Road_traffic_density    5
Type_of_order           4
Type_of_vehicle         4
Festival                3
City                    4
Name: levels incl. missing, dtype: int64

## 9. Target variable — `Time_taken(min)`

In [12]:
y = d["target_min"]
q = y.quantile([.5, .75, .8, .85, .9, .95]).to_dict()
METRICS["target"] = {"format_example": raw["Time_taken(min)"].iloc[0], "min": int(y.min()), "max": int(y.max()),
                     "mean": round(float(y.mean()), 3), "median": float(y.median()), "std": round(float(y.std()), 3),
                     "skew": round(float(y.skew()), 3), "n_unique": int(y.nunique()), "missing": int(y.isna().sum()),
                     "quantiles": {str(k): float(v) for k, v in q.items()}}
print(METRICS["target"])

fig, ax = plt.subplots(figsize=(9, 4))
vc = y.value_counts().sort_index()
ax.bar(vc.index, vc.values, color=BLUE, width=0.8)
ax.axvline(y.median(), color=INK2, lw=1.2, ls="--")
ax.text(y.median() + 0.4, vc.max() * 0.97, f"median {y.median():.0f} min", color=INK2, fontsize=9)
ax.set_xlabel("Time_taken (minutes)"); ax.set_ylabel("orders")
ax.set_title("Target distribution — delivery time (integer minutes, 10–54)")
save(fig, "03_target_distribution.png")

{'format_example': '(min) 24', 'min': 10, 'max': 54, 'mean': 26.295, 'median': 26.0, 'std': 9.384, 'skew': 0.486, 'n_unique': 45, 'missing': 0, 'quantiles': {'0.5': 26.0, '0.75': 32.0, '0.8': 34.0, '0.85': 37.0, '0.9': 40.0, '0.95': 44.0}}


saved figures/03_target_distribution.png


In [13]:
seg = ["Road_traffic_density", "Weatherconditions", "City", "Festival", "multiple_deliveries", "Vehicle_condition"]
METRICS["target_by_segment"] = {c: d.groupby(d[c].astype("string").fillna("<missing>"))["target_min"]
                                 .agg(["count", "mean", "median"]).round(2).to_dict("index") for c in seg}
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, c in zip(axes.flat, seg):
    tmp = d[[c, "target_min"]].copy(); tmp[c] = tmp[c].astype("string").fillna("<missing>")
    order = tmp.groupby(c)["target_min"].median().sort_values().index
    sns.boxplot(data=tmp, x=c, y="target_min", order=order, color="#9cc3f0", linewidth=1,
                fliersize=1.5, ax=ax)
    ax.set_title(f"Target by {c}"); ax.set_xlabel(""); ax.set_ylabel("minutes")
    ax.tick_params(axis="x", rotation=25)
fig.suptitle("Delivery time by segment (descriptive, not causal)", fontweight="bold", color=INK)
save(fig, "06_target_by_segment.png")

saved figures/06_target_by_segment.png


## 10. Geographic validity

In [14]:
rlat, rlon = d.Restaurant_latitude, d.Restaurant_longitude
dlat, dlon = d.Delivery_location_latitude, d.Delivery_location_longitude
INDIA = dict(lat=(6.0, 37.5), lon=(68.0, 97.5))  # generous mainland-India bounding box (assumption)
in_india = lambda la, lo: la.between(*INDIA["lat"]) & lo.between(*INDIA["lon"])
zero_rest = (rlat == 0) & (rlon == 0)
neg_rest = (rlat < 0) | (rlon < 0)
geo = {"restaurant_zero_coords": int(zero_rest.sum()),
       "restaurant_negative_lat": int((rlat < 0).sum()), "restaurant_negative_lon": int((rlon < 0).sum()),
       "restaurant_any_negative": int(neg_rest.sum()),
       "restaurant_in_india_raw": int(in_india(rlat, rlon).sum()),
       "restaurant_in_india_after_abs": int(in_india(rlat.abs(), rlon.abs()).sum()),
       "delivery_in_india_raw": int(in_india(dlat, dlon).sum()),
       "delivery_near_zero(<1deg)": int(((dlat.abs() < 1) & (dlon.abs() < 1)).sum()),
       "delivery_near_zero_coincides_with_zero_restaurant": bool((((dlat.abs() < 1) & (dlon.abs() < 1)) == zero_rest).all()),
       "negative_restaurant_rows_with_positive_delivery": int((neg_rest & (dlat > 0) & (dlon > 0)).sum()),
       "unique_restaurant_locations_abs": int(d.assign(a=rlat.abs(), b=rlon.abs()).groupby(["a", "b"]).ngroups)}

off_lat = (dlat - rlat.abs()).round(6); off_lon = (dlon - rlon.abs()).round(6)
valid = in_india(rlat.abs(), rlon.abs()) & in_india(dlat, dlon)
geo["valid_rows_after_abs"] = int(valid.sum())
geo["lat_offset_equals_lon_offset_share"] = round(float((off_lat == off_lon)[valid].mean()), 4)
geo["lat_offset_values_top"] = {str(k): int(v) for k, v in off_lat[valid].value_counts().head(15).items()}

def haversine(la1, lo1, la2, lo2):
    la1, lo1, la2, lo2 = map(np.radians, (la1, lo1, la2, lo2))
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(h))
d["dist_km_audit"] = np.where(valid, haversine(rlat.abs(), rlon.abs(), dlat, dlon), np.nan)
geo["distance_km_valid"] = d["dist_km_audit"].describe().round(3).to_dict()
geo["pearson_distance_target_valid"] = round(float(d.loc[valid, ["dist_km_audit", "target_min"]].corr().iloc[0, 1]), 4)
METRICS["geo"] = geo
geo

{'restaurant_zero_coords': 3640,
 'restaurant_negative_lat': 431,
 'restaurant_negative_lon': 162,
 'restaurant_any_negative': 431,
 'restaurant_in_india_raw': 41522,
 'restaurant_in_india_after_abs': 41953,
 'delivery_in_india_raw': 41953,
 'delivery_near_zero(<1deg)': 3640,
 'delivery_near_zero_coincides_with_zero_restaurant': True,
 'negative_restaurant_rows_with_positive_delivery': 431,
 'unique_restaurant_locations_abs': 389,
 'valid_rows_after_abs': 41953,
 'lat_offset_equals_lon_offset_share': 0.9822,
 'lat_offset_values_top': {'0.05': 3775,
  '0.02': 3761,
  '0.09': 3759,
  '0.07': 3740,
  '0.11': 3728,
  '0.01': 3698,
  '0.04': 3685,
  '0.03': 3678,
  '0.08': 3654,
  '0.06': 3641,
  '0.13': 3622,
  '0.14': 551,
  '0.130001': 73,
  '0.060001': 61,
  '0.080001': 58},
 'distance_km_valid': {'count': 41953.0,
  'mean': 9.72,
  'std': 5.603,
  'min': 1.465,
  '25%': 4.658,
  '50%': 9.193,
  '75%': 13.681,
  'max': 20.969},
 'pearson_distance_target_valid': 0.3205}

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ax = axes[0]
ax.scatter(rlon[~neg_rest & ~zero_rest], rlat[~neg_rest & ~zero_rest], s=6, color=BLUE, alpha=.5, label="in range")
ax.scatter(rlon[neg_rest], rlat[neg_rest], s=10, color=ORANGE, label=f"negative sign ({neg_rest.sum():,})")
ax.scatter(rlon[zero_rest], rlat[zero_rest], s=30, color=INK, marker="x", label=f"(0, 0) ({zero_rest.sum():,})")
ax.set_title("Restaurant coordinates — raw"); ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.legend(fontsize=8, loc="lower left")
ax = axes[1]
ax.scatter(dlon[valid], dlat[valid], s=4, color=AQUA, alpha=.35, label="delivery")
ax.scatter(rlon.abs()[valid], rlat.abs()[valid], s=8, color=BLUE, alpha=.6, label="restaurant (|sign| fixed)")
ax.set_title(f"Valid pairs after sign fix (n={valid.sum():,})"); ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.legend(fontsize=8)
ax = axes[2]
ax.hist(d["dist_km_audit"].dropna(), bins=50, color=BLUE, edgecolor=SURF, linewidth=.5)
ax.set_title("Haversine distance, valid pairs"); ax.set_xlabel("km"); ax.set_ylabel("orders")
fig.suptitle("Geographic data quality", fontweight="bold", color=INK)
save(fig, "07_geographic_distribution.png")

saved figures/07_geographic_distribution.png


## 11. Date & time columns

In [16]:
gap = (d.t_picked - d.t_ordered).dt.total_seconds() / 60
tinfo = {"order_date_min": str(d.order_date.min().date()), "order_date_max": str(d.order_date.max().date()),
         "n_unique_dates": int(d.order_date.nunique()),
         "calendar_dates_without_orders": [str(x.date()) for x in pd.date_range(d.order_date.min(), d.order_date.max())
                                           .difference(d.order_date.dropna().unique())],
         "rows_per_month": {str(k): int(v) for k, v in d.order_date.dt.to_period("M").value_counts().sort_index().items()},
         "rows_per_weekday": d.order_date.dt.day_name().value_counts().to_dict(),
         "Time_Orderd_missing": int(d.t_ordered.isna().sum()),
         "pickup_minus_order_minutes_counts": {str(k): int(v) for k, v in gap.value_counts().items()},
         "negative_gap_rows (midnight crossing)": int((gap < 0).sum()),
         "orders_by_hour": {int(k): int(v) for k, v in d.t_ordered.dt.hour.value_counts().sort_index().items()}}
METRICS["time"] = tinfo
tinfo

{'order_date_min': '2022-02-11',
 'order_date_max': '2022-04-06',
 'n_unique_dates': 44,
 'calendar_dates_without_orders': ['2022-02-19',
  '2022-02-20',
  '2022-02-21',
  '2022-02-22',
  '2022-02-23',
  '2022-02-24',
  '2022-02-25',
  '2022-02-26',
  '2022-02-27',
  '2022-02-28',
  '2022-03-22'],
 'rows_per_month': {'2022-02': 7242, '2022-03': 31989, '2022-04': 6362},
 'rows_per_weekday': {'Wednesday': 7093,
  'Friday': 7031,
  'Tuesday': 6375,
  'Thursday': 6348,
  'Saturday': 6290,
  'Sunday': 6249,
  'Monday': 6207},
 'Time_Orderd_missing': 1731,
 'pickup_minus_order_minutes_counts': {'5.0': 14564,
  '10.0': 14288,
  '15.0': 14179,
  '-1425.0': 431,
  '-1430.0': 261,
  '-1435.0': 139},
 'negative_gap_rows (midnight crossing)': 831,
 'orders_by_hour': {0: 430,
  8: 1818,
  9: 1947,
  10: 1991,
  11: 1962,
  12: 892,
  13: 784,
  14: 791,
  15: 873,
  16: 709,
  17: 4278,
  18: 4480,
  19: 4595,
  20: 4539,
  21: 4686,
  22: 4576,
  23: 4511}}

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
h = pd.Series(tinfo["orders_by_hour"]).reindex(range(24), fill_value=0)
axes[0].bar(h.index, h.values, color=BLUE, width=.75)
axes[0].set_xticks(range(0, 24, 2)); axes[0].set_xlabel("hour of Time_Orderd"); axes[0].set_ylabel("orders")
axes[0].set_title("Orders by hour (no orders 01:00–07:59)")
daily = d.groupby("order_date").size()
axes[1].bar(daily.index, daily.values, color=BLUE, width=0.8)
axes[1].set_title(f"Orders per date ({daily.size} dates; gaps = no data)"); axes[1].set_ylabel("orders"); axes[1].tick_params(axis="x", rotation=30)
save(fig, "08_time_patterns.png")

saved figures/08_time_patterns.png


## 12. Outliers & invalid values

In [18]:
def iqr_bounds(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - 1.5 * i, q3 + 1.5 * i
out = {}
for c in ["Delivery_person_Age", "Delivery_person_Ratings", "target_min", "dist_km_audit", "multiple_deliveries"]:
    s = d[c].dropna(); lo, hi = iqr_bounds(s)
    out[c] = {"iqr_low": round(float(lo), 3), "iqr_high": round(float(hi), 3),
              "n_below": int((s < lo).sum()), "n_above": int((s > hi).sum())}
inv = {"ratings_above_5": int((d.Delivery_person_Ratings > 5).sum()),
       "ratings_eq_1": int((d.Delivery_person_Ratings == 1).sum()),
       "age_15 (below typical working age)": int((d.Delivery_person_Age == 15).sum()),
       "age_50": int((d.Delivery_person_Age == 50).sum()),
       "age15_rows_all_have_rating_1": bool((d.loc[d.Delivery_person_Age == 15, "Delivery_person_Ratings"] == 1).all()),
       "age50_rows_all_have_rating_6": bool((d.loc[d.Delivery_person_Age == 50, "Delivery_person_Ratings"] == 6).all()),
       "vehicle_condition_3": int((d.Vehicle_condition == 3).sum())}
METRICS["outliers_iqr"] = out; METRICS["invalid_values"] = inv
display(pd.DataFrame(out).T); inv

,iqr_low,iqr_high,n_below,n_above
Delivery_person_Age,10.000,50.000,0.0,0.0
Delivery_person_Ratings,3.900,5.500,1170.0,53.0
target_min,-0.500,51.500,0.0,270.0
dist_km_audit,-8.877,27.216,0.0,0.0
multiple_deliveries,-1.500,2.500,0.0,361.0


{'ratings_above_5': 53,
 'ratings_eq_1': 38,
 'age_15 (below typical working age)': 38,
 'age_50': 53,
 'age15_rows_all_have_rating_1': True,
 'age50_rows_all_have_rating_6': True,
 'vehicle_condition_3': 520}

In [19]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4.2))
for ax, c in zip(axes, ["Delivery_person_Age", "Delivery_person_Ratings", "target_min", "dist_km_audit"]):
    sns.boxplot(y=d[c].dropna(), color="#9cc3f0", linewidth=1, fliersize=2, ax=ax,
                flierprops=dict(markerfacecolor=ORANGE, markeredgecolor=ORANGE))
    o = out[c]; ax.set_title(c, fontsize=10); ax.set_ylabel("")
    ax.set_xlabel(f"IQR outliers: {o['n_below'] + o['n_above']:,}", color=INK2)
fig.suptitle("Outlier analysis (1.5×IQR rule)", fontweight="bold", color=INK)
save(fig, "09_outliers.png")

saved figures/09_outliers.png


## 13. Correlation analysis (numeric fields + audit distance)

In [20]:
ccols = ["Delivery_person_Age", "Delivery_person_Ratings", "Vehicle_condition", "multiple_deliveries",
         "dist_km_audit", "target_min"]
corr = d[ccols].corr(method="spearman").round(2)
METRICS["spearman_corr"] = corr.to_dict()
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap=DIV, vmin=-1, vmax=1, center=0, square=True,
            linewidths=2, linecolor=SURF, cbar_kws={"shrink": .7}, ax=ax)
ax.set_title("Spearman correlation (pairwise complete rows)")
save(fig, "10_correlation.png")
corr

saved figures/10_correlation.png


,Delivery_person_Age,Delivery_person_Ratings,Vehicle_condition,multiple_deliveries,dist_km_audit,target_min
Delivery_person_Age,1.00,-0.10,-0.00,0.11,-0.00,0.31
Delivery_person_Ratings,-0.10,1.00,0.07,-0.09,-0.07,-0.29
Vehicle_condition,-0.00,0.07,1.00,-0.09,0.01,-0.22
multiple_deliveries,0.11,-0.09,-0.09,1.00,0.12,0.34
dist_km_audit,-0.00,-0.07,0.01,0.12,1.00,0.32
target_min,0.31,-0.29,-0.22,0.34,0.32,1.00


## 14. Rider identifier & `City` semantics
Check whether `Delivery_person_ID` identifies a single person (needed for rider segmentation),
and what `City` actually encodes.

In [21]:
g = d.groupby("Delivery_person_ID")
rid_city = d.Delivery_person_ID.str.extract(r"^([A-Z]+)RES")[0]
rider = {"n_ids": int(d.Delivery_person_ID.nunique()),
         "orders_per_id": g.size().describe().round(2).to_dict(),
         "ids_with_more_than_one_age": int((g.Delivery_person_Age.nunique() > 1).sum()),
         "distinct_ages_per_id": g.Delivery_person_Age.nunique().describe().round(2).to_dict(),
         "ids_with_more_than_one_vehicle_type": int((g.Type_of_vehicle.nunique() > 1).sum()),
         "restaurant_locations_per_id_max": int(g.apply(lambda x: x[["Restaurant_latitude", "Restaurant_longitude"]]
                                                       .abs().drop_duplicates().shape[0], include_groups=False).max()),
         "id_pattern_ok": bool(d.Delivery_person_ID.str.fullmatch(r"[A-Z]+RES\d+DEL\d+").all()),
         "city_codes_in_id": int(rid_city.nunique()),
         "orders_per_city_code": rid_city.value_counts().to_dict(),
         "City_column_levels": d.City.value_counts(dropna=False).to_dict()}
vc_type = pd.crosstab(d.Vehicle_condition, d.Type_of_vehicle)
rider["vehicle_condition_by_type"] = vc_type.to_dict()
METRICS["rider_id"] = rider
display(vc_type)
rider

Type_of_vehicle,bicycle,electric_scooter,motorcycle,scooter
Vehicle_condition,,,,
0,0,0,15009,0
1,0,0,7583,7447
2,0,3626,3753,7655
3,68,188,90,174


{'n_ids': 1320,
 'orders_per_id': {'count': 1320.0,
  'mean': 34.54,
  'std': 21.31,
  'min': 5.0,
  '25%': 13.0,
  '50%': 41.0,
  '75%': 56.0,
  'max': 67.0},
 'ids_with_more_than_one_age': 1320,
 'distinct_ages_per_id': {'count': 1320.0,
  'mean': 14.22,
  'std': 4.97,
  'min': 4.0,
  '25%': 9.0,
  '50%': 17.0,
  '75%': 19.0,
  'max': 22.0},
 'ids_with_more_than_one_vehicle_type': 1320,
 'restaurant_locations_per_id_max': 1,
 'id_pattern_ok': True,
 'city_codes_in_id': 22,
 'orders_per_city_code': {'JAP': 3443,
  'RANCHI': 3229,
  'BANG': 3195,
  'SUR': 3187,
  'HYD': 3181,
  'MUM': 3173,
  'MYS': 3171,
  'COIMB': 3170,
  'VAD': 3166,
  'INDO': 3159,
  'CHEN': 3145,
  'PUNE': 3132,
  'AGR': 763,
  'LUDH': 758,
  'KNP': 740,
  'ALH': 740,
  'DEH': 737,
  'GOA': 709,
  'AURG': 703,
  'KOC': 701,
  'KOL': 700,
  'BHP': 691},
 'City_column_levels': {'Metropolitian': 34093,
  'Urban': 10136,
  nan: 1200,
  'Semi-Urban': 164},
 'vehicle_condition_by_type': {'bicycle': {0: 0, 1: 0, 2: 0, 3:

## 15. Persist audit metrics & verify the raw file is untouched

In [22]:
RAW_SHA_END = sha256(RAW)
METRICS["raw_sha256"] = RAW_SHA_END
METRICS["raw_unchanged"] = RAW_SHA_START == RAW_SHA_END
assert METRICS["raw_unchanged"], "RAW FILE CHANGED — investigate immediately"
with open(REP / "audit_metrics.json", "w", encoding="utf-8") as f:
    json.dump(METRICS, f, indent=2, default=str)
print("raw file unchanged:", METRICS["raw_unchanged"], "| wrote reports/audit_metrics.json")

raw file unchanged: True | wrote reports/audit_metrics.json
